### RAG Pipelines - Data Ingestion to Vector DB Pipeline

In [5]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [6]:
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

Found 3 PDF files to process

Processing: Agile_Handbook_Batch3_Kanban_XP_Lean-15.pdf
  ✓ Loaded 11 pages

Processing: Agile_Handbook_Batch3_Kanban_XP_Lean-17.pdf
  ✓ Loaded 11 pages

Processing: Agile_Handbook_Batch3_Kanban_XP_Lean-11.pdf
  ✓ Loaded 11 pages

Total documents loaded: 33


In [7]:
all_pdf_documents

[Document(metadata={'producer': 'PyPDF', 'creator': 'Google', 'creationdate': '', 'source': '../data/pdf/Agile_Handbook_Batch3_Kanban_XP_Lean-15.pdf', 'total_pages': 11, 'page': 0, 'page_label': '1', 'source_file': 'Agile_Handbook_Batch3_Kanban_XP_Lean-15.pdf', 'file_type': 'pdf'}, page_content='Agile Handbook – Batch 3\nKanban, XP, and Lean Software \nDevelopment'),
 Document(metadata={'producer': 'PyPDF', 'creator': 'Google', 'creationdate': '', 'source': '../data/pdf/Agile_Handbook_Batch3_Kanban_XP_Lean-15.pdf', 'total_pages': 11, 'page': 1, 'page_label': '2', 'source_file': 'Agile_Handbook_Batch3_Kanban_XP_Lean-15.pdf', 'file_type': 'pdf'}, page_content='Kanban Overview\n• Visual workflow management method\n• Uses a Kanban board with columns (To Do, In \nProgress, Done)\n• Focuses on continuous delivery and flow \nefficiency\n• Helps teams visualize work and limit work in \nprogress'),
 Document(metadata={'producer': 'PyPDF', 'creator': 'Google', 'creationdate': '', 'source': '../d

In [8]:
### Text Splitting to get chunks

def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    """Split documents into smaller chunks"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [9]:
chunks=split_documents(all_pdf_documents)
chunks

Split 33 documents into 33 chunks

Example chunk:
Content: Agile Handbook – Batch 3
Kanban, XP, and Lean Software 
Development...
Metadata: {'producer': 'PyPDF', 'creator': 'Google', 'creationdate': '', 'source': '../data/pdf/Agile_Handbook_Batch3_Kanban_XP_Lean-15.pdf', 'total_pages': 11, 'page': 0, 'page_label': '1', 'source_file': 'Agile_Handbook_Batch3_Kanban_XP_Lean-15.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'PyPDF', 'creator': 'Google', 'creationdate': '', 'source': '../data/pdf/Agile_Handbook_Batch3_Kanban_XP_Lean-15.pdf', 'total_pages': 11, 'page': 0, 'page_label': '1', 'source_file': 'Agile_Handbook_Batch3_Kanban_XP_Lean-15.pdf', 'file_type': 'pdf'}, page_content='Agile Handbook – Batch 3\nKanban, XP, and Lean Software \nDevelopment'),
 Document(metadata={'producer': 'PyPDF', 'creator': 'Google', 'creationdate': '', 'source': '../data/pdf/Agile_Handbook_Batch3_Kanban_XP_Lean-15.pdf', 'total_pages': 11, 'page': 1, 'page_label': '2', 'source_file': 'Agile_Handbook_Batch3_Kanban_XP_Lean-15.pdf', 'file_type': 'pdf'}, page_content='Kanban Overview\n• Visual workflow management method\n• Uses a Kanban board with columns (To Do, In \nProgress, Done)\n• Focuses on continuous delivery and flow \nefficiency\n• Helps teams visualize work and limit work in \nprogress'),
 Document(metadata={'producer': 'PyPDF', 'creator': 'Google', 'creationdate': '', 'source': '../d

### Embedding and  vectorStroreDB

In [10]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

/Users/arjun/rag/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [11]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


## initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6656.29it/s]


Model loaded successfully. Embedding dimension: 384


/var/folders/l2/t145cxk97blf2g7c_4j6jv640000gn/T/ipykernel_66390/2964522620.py:20: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")


### VectorStore

In [12]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 0


In [13]:
chunks

[Document(metadata={'producer': 'PyPDF', 'creator': 'Google', 'creationdate': '', 'source': '../data/pdf/Agile_Handbook_Batch3_Kanban_XP_Lean-15.pdf', 'total_pages': 11, 'page': 0, 'page_label': '1', 'source_file': 'Agile_Handbook_Batch3_Kanban_XP_Lean-15.pdf', 'file_type': 'pdf'}, page_content='Agile Handbook – Batch 3\nKanban, XP, and Lean Software \nDevelopment'),
 Document(metadata={'producer': 'PyPDF', 'creator': 'Google', 'creationdate': '', 'source': '../data/pdf/Agile_Handbook_Batch3_Kanban_XP_Lean-15.pdf', 'total_pages': 11, 'page': 1, 'page_label': '2', 'source_file': 'Agile_Handbook_Batch3_Kanban_XP_Lean-15.pdf', 'file_type': 'pdf'}, page_content='Kanban Overview\n• Visual workflow management method\n• Uses a Kanban board with columns (To Do, In \nProgress, Done)\n• Focuses on continuous delivery and flow \nefficiency\n• Helps teams visualize work and limit work in \nprogress'),
 Document(metadata={'producer': 'PyPDF', 'creator': 'Google', 'creationdate': '', 'source': '../d

In [14]:
### Convert the text to embeddings
texts=[doc.page_content for doc in chunks]

## Generate the Embeddings

embeddings=embedding_manager.generate_embeddings(texts)

##store int he vector dtaabase
vectorstore.add_documents(chunks,embeddings)

Generating embeddings for 33 texts...


Batches: 100%|██████████| 2/2 [00:02<00:00,  1.46s/it]

Generated embeddings with shape: (33, 384)
Adding 33 documents to vector store...
Successfully added 33 documents to vector store
Total documents in collection: 33


### Retriever Pipeline 


In [ ]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)

In [16]:
rag_retriever.retrieve("What is LEAN")

Retrieving documents for query: 'What is LEAN'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  1.41it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_5d794d0a_8',
  'content': 'Lean Software Development \nPrinciples\n• Eliminate waste\n• Amplify learning\n• Decide as late as possible\n• Deliver as fast as possible\n• Empower the team\n• Build integrity in\n• See the whole',
  'metadata': {'page': 8,
   'source': '../data/pdf/Agile_Handbook_Batch3_Kanban_XP_Lean-15.pdf',
   'source_file': 'Agile_Handbook_Batch3_Kanban_XP_Lean-15.pdf',
   'creationdate': '',
   'file_type': 'pdf',
   'content_length': 189,
   'creator': 'Google',
   'total_pages': 11,
   'page_label': '9',
   'producer': 'PyPDF',
   'doc_index': 8},
  'similarity_score': 0.16177403926849365,
  'distance': 0.8382259607315063,
  'rank': 1},
 {'id': 'doc_fee7b054_19',
  'content': 'Lean Software Development \nPrinciples\n• Eliminate waste\n• Amplify learning\n• Decide as late as possible\n• Deliver as fast as possible\n• Empower the team\n• Build integrity in\n• See the whole',
  'metadata': {'page': 8,
   'content_length': 189,
   'source_file': 'Agile_Hand

### Integrate VectorDB context with LLM 


In [27]:
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()  # Load environment variables from .env file  

groq_api_key = os.getenv("GROQ_API_KEY")

llm=ChatGroq(
    api_key=groq_api_key,
    model="llama-3.1-8b-instant",
    temperature=0.1,
    max_tokens=1024
)

def rag_simple(query,retriever,llm,top_k=3):
    results=retriever.retrieve(query,top_k=top_k)
    context = "\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant documents found."
    
     ## generate ans
    prompt=f"""Using the following context, answer the question concisely.
      Context: {context}
      Question: {query}
      Answer:"""
    response=llm.invoke(prompt.format(context=context, query=query))
    return response.content



In [28]:
answer=rag_simple("What is LEAN",rag_retriever,llm)
print(answer)

Retrieving documents for query: 'What is LEAN'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  4.02it/s]


Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)
Lean is a software development methodology that aims to maximize value for customers while minimizing waste, focusing on continuous improvement and rapid delivery.


In [32]:
### advanced rag

# --- Enhanced RAG Pipeline Features ---
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}
    
    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])
    
    # Generate answer
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])
    
    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output

# Example usage:
result = rag_advanced("Kanban", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])

Retrieving documents for query: 'Kanban'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  2.40it/s]


Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)
Answer: Kanban is a visual system for managing work, commonly used in IT Operations and Support Teams, Marketing and Content Production, HR Recruitment Pipelines, Maintenance and Bug Fixing Work, and Continuous Delivery Environments.
Sources: [{'source': 'Agile_Handbook_Batch3_Kanban_XP_Lean-15.pdf', 'page': 4, 'score': 0.11909347772598267, 'preview': 'Kanban Real-world Use Cases\n• IT Operations and Support Teams\n• Marketing and Content Production\n• HR Recruitment Pipelines\n• Maintenance and Bug Fixing Work\n• Continuous Delivery Environments...'}, {'source': 'Agile_Handbook_Batch3_Kanban_XP_Lean-17.pdf', 'page': 4, 'score': 0.11909347772598267, 'preview': 'Kanban Real-world Use Cases\n• IT Operations and Support Teams\n• Marketing and Content Production\n• HR Recruitment Pipelines\n• Maintenance and Bug Fixing Work\n• Continuous Delivery Environments...'}, {'source': 'Agile_Handbook_Batch3_Kanban_XP_

In [34]:
# --- Advanced RAG Pipeline: Streaming, Citations, History, Summarization ---
from typing import List, Dict, Any
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []  # Store query history

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        # Retrieve relevant documents
        results = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)
        if not results:
            answer = "No relevant context found."
            sources = []
            context = ""
        else:
            context = "\n\n".join([doc['content'] for doc in results])
            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page': doc['metadata'].get('page', 'unknown'),
                'score': doc['similarity_score'],
                'preview': doc['content'][:120] + '...'
            } for doc in results]
            # Streaming answer simulation
            prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"""
            if stream:
                print("Streaming answer:")
                for i in range(0, len(prompt), 80):
                    print(prompt[i:i+80], end='', flush=True)
                    time.sleep(0.05)
                print()
            response = self.llm.invoke([prompt.format(context=context, question=question)])
            answer = response.content

        # Add citations to answer
        citations = [f"[{i+1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)]
        answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer

        # Optionally summarize answer
        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary_resp = self.llm.invoke([summary_prompt])
            summary = summary_resp.content

        # Store query history
        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })

        return {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }

# Example usage:
adv_rag = AdvancedRAGPipeline(rag_retriever, llm)
result = adv_rag.query("what is agile", top_k=3, min_score=0.1, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])

Retrieving documents for query: 'what is agile'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  1.94it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)
Streaming answer:
Use the following context to answer the question concisely.
Context:
Extreme Programming (XP) 
Overview
• Agile methodology focused on engineering 
practices
• Emphasizes technical excellence and frequent 
releases
• Core values: Communication, Simplicity, 
Feedback, Courage, Respect

Extreme Programming (XP) 
Overview


• Agile methodology focused on engineering 
practices
• Emphasizes technical excellence and frequent 
releases
• Core values: Communication, Simplicity, 
Feedback, Courage, Respect

Extreme Programming (XP) 
Overview
• Agile methodology focused on engineering 
practices
• Emphasizes technical excellence and frequent 
releases
• Core values: Communication, Simplicity, 
Feedback, Courage, Respect

Question: what is agile

Answer:

Final Answer: Agile is an iterative and flexible approach to project management that emphasizes collaboration, speed, and continuous improvement.

Citations:
[1] Agile_Handbook_Batch3_Kanban_XP_Lean-15.pdf (page 5)
[2] Agile_Handbook_Batch3_Kanban_XP_Lean-17.pdf (page 5)
[3] Agile_Handbook_Batch3_Kanban_XP_Lean-11.pdf (page 5)
Summary: Agile is a project management approach that emphasizes collaboration, speed, and continuous improvement through iterative and flexible methods. It focuses on adapting to changing requirements and delivering results quickly, with